https://coinmarketcap.com/rankings/exchanges/

## workflow

For each run date, the dataset is constructed using a strict day-by-day filtering workflow so that only eligible assets are stored in the final tables.

### Step 1: Retrieve the daily top-100 candidate universe
For each date in the selected run period, the script requests market data from CoinGecko using `vs_currency = usd`, `order = market_cap_desc`, and `per_page = 100`. The returned result is used as the daily top-100 candidate universe for that run date.

The following information is taken from the CoinGecko markets response:
- `coingecko_id`
- `symbol`
- `name`
- `current_price`
- `market_cap`
- `total_volume`
- `circulating_supply`
- `total_supply`
- `max_supply`

### Step 2: Apply the daily volume filter
From the daily top-100 candidate universe, only assets with a 24-hour trading volume above USD 50 million are retained. The volume measure used for this filtering step is CoinGecko’s `total_volume` field, which is stored in the dataset as `global_volume_24h_usd`.

This means that an asset is only considered further if:

`global_volume_24h_usd > 50,000,000`

### Step 3: Determine top-exchange coverage
For the assets that remain after the daily volume filter, the number of major exchanges on which each asset is traded is determined.

The major exchanges are defined as the top `TOP_EXCHANGES_N` exchanges taken from the CoinMarketCap spot exchange rankings page at run time. These exchange names are then matched to CoinGecko exchange identifiers using the CoinGecko `/exchanges/list` endpoint.

For each candidate asset:
- if the asset is already present in the asset master table and already has a stored `top_exchange_count`, that value is reused
- if the asset is not yet in the asset master table, or its `top_exchange_count` is missing, the value is fetched from the CoinGecko `/coins/{id}/tickers` endpoint by counting how many of the selected major exchanges appear among the asset’s trading venues

### Step 4: Apply the exchange coverage filter
Assets are then filtered by exchange coverage. Only assets that are traded on at least 10 major exchanges are retained.

This means that an asset is only stored if:

`top_exchange_count >= 10`

### Step 5: Construct the final eligible asset set for the day
After both filters have been applied, the remaining assets for the day satisfy all of the following conditions:
1. they are in the daily top 100 candidate universe used by the script
2. they have more than USD 50 million in 24-hour trading volume
3. they are traded on at least 10 major exchanges

Only these assets are allowed to enter the stored dataset.

### Step 6: Update the asset master table
The surviving assets are merged into the `asset` master table. This table stores one row per eligible asset and contains relatively stable asset-level metadata.

The `asset` table contains:
- `asset_id` (internal stable identifier)
- `coingecko_id`
- `symbol`
- `name`
- `circulating_supply`
- `total_supply`
- `max_supply`
- `top_exchange_count`

New eligible assets receive a new `asset_id`. Existing eligible assets keep their previously assigned `asset_id`.

### Step 7: Create the daily asset table
For the same surviving assets, one daily observation per `(date, asset_id)` is written to the `asset_daily` table.

The `asset_daily` table contains:
- `date`
- `asset_id`
- `price_usd`
- `market_cap_usd`
- `global_volume_24h_usd`
- `binance_volume_24h_usd`

In the current workflow, the fields `price_usd`, `market_cap_usd`, and `global_volume_24h_usd` are taken directly from the CoinGecko markets response for the respective run date:
- `current_price -> price_usd`
- `market_cap -> market_cap_usd`
- `total_volume -> global_volume_24h_usd`

The column `binance_volume_24h_usd` is currently not used for filtering and may remain empty (`NA`) at this stage.

### Step 8: Repeat for all run dates
This workflow is repeated independently for every date in the requested run range. As a result, the final dataset is built as a filtered daily panel in which only assets meeting all selection criteria on a given day are stored.

## Resulting table semantics

### `asset`
The `asset` table is an asset master table. It stores one row per asset that has passed the selection criteria on at least one processed date.

### `asset_daily`
The `asset_daily` table is a daily panel table. It stores one row per eligible asset-date combination and contains the daily market variables used in the analysis.

## Rationale
This workflow ensures that:
- filtering is applied before storage
- only relevant assets enter the persistent dataset
- the asset master table contains only eligible assets
- the daily table contains only observations that satisfy the study’s inclusion criteria
- unnecessary storage of ineligible assets is avoided

In [70]:
# Imports
from pathlib import Path
import time
import configparser
from datetime import datetime

import pandas as pd
import requests

In [71]:
# Configuration
START_DATE = "2026-03-01"
END_DATE = "2026-03-31"  # inclusive
VS_CURRENCY = "usd"

TOP_N = 100
TOP_EXCHANGES_N = 20

MIN_TOP_EXCHANGE_COUNT = 10
MIN_VOLUME_24H_USD = 50_000_000

DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily.parquet"

COINGECKO_BASE_URL = "https://api.coingecko.com/api/v3"
REQUEST_SLEEP_SECONDS = 2.05
REQUEST_TIMEOUT_SECONDS = 30
USER_AGENT = "TDA-Data-Aquisition"

config = configparser.ConfigParser()
config.read("../../secret.cfg")

COINGECKO_API_KEY = config.get("COINGECKO", "DEMO_API_KEY")

if not COINGECKO_API_KEY:
    raise ValueError("CoinGecko DEMO_API_KEY not found in secret.cfg")

In [72]:
# Table schemas
ASSET_COLUMNS = [
    "asset_id",
    "coingecko_id",
    "symbol",
    "name",
    "circulating_supply",
    "total_supply",
    "max_supply",
    "top_exchange_count",
]

ASSET_DAILY_COLUMNS = [
    "date",
    "asset_id",
    "price_usd",
    "market_cap_usd",
    "global_volume_24h_usd",
    "binance_volume_24h_usd",
]

In [73]:
# Request helpers
def coingecko_get(endpoint: str, params: dict | None = None, max_retries: int = 5) -> dict | list:
    url = f"{COINGECKO_BASE_URL}{endpoint}"
    headers = {
        "User-Agent": USER_AGENT,
        "x-cg-demo-api-key": COINGECKO_API_KEY,
    }
    params = dict(params or {})
    last_error = None

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        if response.status_code == 429:
            wait_seconds = min(90, 5 * (attempt + 1))
            print(
                f"429 for {endpoint} | attempt {attempt + 1}/{max_retries} | sleeping {wait_seconds}s"
            )
            time.sleep(wait_seconds)
            last_error = RuntimeError(f"429 rate limit for {url}")
            continue

        response.raise_for_status()
        time.sleep(REQUEST_SLEEP_SECONDS)
        return response.json()

    raise last_error or RuntimeError(f"Request failed for {url}")

In [74]:
# API test
print(coingecko_get("/ping"))

{'gecko_says': '(V3) To the Moon!'}


In [75]:
# Validation and storage helpers
def ensure_output_dirs() -> None:
    FETCHED_DIR.mkdir(parents=True, exist_ok=True)


def enforce_columns(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected columns: {extra}")

    return df[expected_columns].copy()


def ensure_asset_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "top_exchange_count" not in df.columns:
        df["top_exchange_count"] = pd.NA

    return enforce_columns(df, ASSET_COLUMNS)


def validate_date_range(start_date: str, end_date: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)

    if start_dt > end_dt:
        raise ValueError(f"START_DATE must be on or before END_DATE. Got {start_date} > {end_date}.")

    today_utc = pd.Timestamp.now(tz="UTC").normalize()
    last_finished_day = today_utc - pd.Timedelta(days=1)

    if start_dt.tz_localize("UTC") > last_finished_day:
        raise ValueError(
            f"START_DATE must be a finished UTC day. Latest allowed date is {last_finished_day.date()}."
        )

    if end_dt.tz_localize("UTC") > last_finished_day:
        raise ValueError(
            f"END_DATE must be a finished UTC day. Latest allowed date is {last_finished_day.date()}."
        )

    return start_dt, end_dt


def make_date_list(start_date: str, end_date: str) -> list[str]:
    start_dt, end_dt = validate_date_range(start_date, end_date)
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(start=start_dt, end=end_dt, freq="D")]


def load_existing_assets() -> pd.DataFrame:
    if ASSETS_PATH.exists():
        df = pd.read_parquet(ASSETS_PATH)
        return ensure_asset_columns(df)
    return pd.DataFrame(columns=ASSET_COLUMNS)


def load_existing_asset_daily() -> pd.DataFrame:
    if ASSET_DAILY_PATH.exists():
        df = pd.read_parquet(ASSET_DAILY_PATH)
        return enforce_columns(df, ASSET_DAILY_COLUMNS)
    return pd.DataFrame(columns=ASSET_DAILY_COLUMNS)

In [76]:
# Coingecko market-data helpers
def to_coingecko_history_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.strftime("%d-%m-%Y")


def fetch_top_n_universe_for_date(date_str: str, top_n: int) -> pd.DataFrame:
    data = coingecko_get(
        "/coins/markets",
        params={
            "vs_currency": VS_CURRENCY,
            "order": "market_cap_desc",
            "per_page": top_n,
            "page": 1,
            "sparkline": "false",
            "price_change_percentage": "24h",
            "date": to_coingecko_history_date(date_str),
        },
    )

    df = pd.DataFrame(data)

    required_cols = [
        "id",
        "symbol",
        "name",
        "current_price",
        "market_cap",
        "total_volume",
        "circulating_supply",
        "total_supply",
        "max_supply",
    ]

    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko response missing required fields: {missing_cols}")

    return df[required_cols].copy()


def prepare_daily_universe_df(top_assets_df: pd.DataFrame, date_str: str) -> pd.DataFrame:
    df = top_assets_df.copy()

    df = df.rename(
        columns={
            "id": "coingecko_id",
            "current_price": "price_usd",
            "market_cap": "market_cap_usd",
            "total_volume": "global_volume_24h_usd",
        }
    )

    df["date"] = date_str
    df["symbol"] = df["symbol"].astype(str).str.upper()

    ordered_cols = [
        "date",
        "coingecko_id",
        "symbol",
        "name",
        "circulating_supply",
        "total_supply",
        "max_supply",
        "price_usd",
        "market_cap_usd",
        "global_volume_24h_usd",
    ]

    return df[ordered_cols].copy()


def filter_daily_universe_by_volume(
    daily_universe_df: pd.DataFrame,
    min_volume_24h_usd: float,
) -> pd.DataFrame:
    df = daily_universe_df.copy()
    df["global_volume_24h_usd"] = pd.to_numeric(df["global_volume_24h_usd"], errors="coerce")

    volume_mask = df["global_volume_24h_usd"] > min_volume_24h_usd
    return df.loc[volume_mask].reset_index(drop=True)

In [77]:
# top exchange hlpers
def fetch_top_exchange_names_from_cmc(top_n: int) -> pd.DataFrame:
    tables = pd.read_html("https://coinmarketcap.com/rankings/exchanges/")
    if not tables:
        raise ValueError("Could not read exchange table from CoinMarketCap rankings page.")

    cmc_df = tables[0].copy()

    name_col = None
    for candidate in cmc_df.columns:
        candidate_str = str(candidate).strip().lower()
        if "exchange" in candidate_str or candidate_str == "name":
            name_col = candidate
            break

    if name_col is None:
        raise ValueError(
            f"Could not find exchange-name column in CoinMarketCap table. Columns: {list(cmc_df.columns)}"
        )

    out = cmc_df[[name_col]].copy()
    out.columns = ["exchange_name"]
    out["exchange_name"] = out["exchange_name"].astype(str).str.strip()
    out = out.loc[out["exchange_name"] != ""].head(top_n).reset_index(drop=True)

    return out


def fetch_coingecko_exchange_id_map() -> pd.DataFrame:
    exchange_list = coingecko_get("/exchanges/list")
    exchange_map_df = pd.DataFrame(exchange_list)

    required_cols = ["id", "name"]
    missing_cols = [c for c in required_cols if c not in exchange_map_df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko exchange list missing columns: {missing_cols}")

    exchange_map_df = exchange_map_df[required_cols].copy()
    exchange_map_df["name_norm"] = (
        exchange_map_df["name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return exchange_map_df


def map_cmc_exchanges_to_coingecko_ids(
    cmc_exchanges_df: pd.DataFrame,
    coingecko_exchange_map_df: pd.DataFrame,
) -> pd.DataFrame:
    cmc_df = cmc_exchanges_df.copy()
    cg_df = coingecko_exchange_map_df.copy()

    cmc_df["name_norm"] = cmc_df["exchange_name"].astype(str).str.strip().str.lower()

    mapped_df = cmc_df.merge(
        cg_df[["id", "name", "name_norm"]],
        on="name_norm",
        how="left",
    )

    mapped_df = mapped_df.rename(
        columns={
            "id": "coingecko_exchange_id",
            "name": "coingecko_exchange_name",
        }
    )

    return mapped_df


def fetch_top_exchange_count_for_asset(coingecko_id: str, exchange_ids: list[str]) -> int:
    if not exchange_ids:
        return 0

    page = 1
    matched_exchange_ids = set()

    while True:
        tickers_data = coingecko_get(
            f"/coins/{coingecko_id}/tickers",
            params={
                "exchange_ids": ",".join(exchange_ids),
                "page": page,
            },
        )

        tickers = tickers_data.get("tickers", [])
        if not tickers:
            break

        for ticker in tickers:
            market = ticker.get("market") or {}
            exchange_id = market.get("identifier")
            if exchange_id in exchange_ids:
                matched_exchange_ids.add(exchange_id)

        if len(tickers) < 100:
            break

        page += 1

    return len(matched_exchange_ids)

In [78]:
# asset table helpers
def build_asset_lookup_by_coingecko_id(asset_df: pd.DataFrame) -> pd.DataFrame:
    asset_df = ensure_asset_columns(asset_df)
    return asset_df[["asset_id", "coingecko_id", "top_exchange_count"]].copy()


def attach_known_top_exchange_counts(
    candidate_df: pd.DataFrame,
    asset_df: pd.DataFrame,
    top_exchange_cache: dict[str, int | float | pd._libs.missing.NAType] | None = None,
) -> pd.DataFrame:
    candidate_df = candidate_df.copy()
    asset_lookup_df = build_asset_lookup_by_coingecko_id(asset_df)

    out = candidate_df.merge(
        asset_lookup_df[["coingecko_id", "top_exchange_count"]],
        on="coingecko_id",
        how="left",
    )

    if top_exchange_cache:
        missing_mask = out["top_exchange_count"].isna()
        out.loc[missing_mask, "top_exchange_count"] = out.loc[missing_mask, "coingecko_id"].map(top_exchange_cache)

    return out


def update_missing_top_exchange_counts_for_candidates(
    candidate_df: pd.DataFrame,
    exchange_ids: list[str],
    exchange_error_rows: list[dict],
    top_exchange_cache: dict[str, int | float | pd._libs.missing.NAType] | None = None,
) -> tuple[pd.DataFrame, int]:
    df = candidate_df.copy()

    if "top_exchange_count" not in df.columns:
        df["top_exchange_count"] = pd.NA

    missing_mask = df["top_exchange_count"].isna()
    fetched_now_count = 0

    for idx in df.index[missing_mask]:
        coingecko_id = str(df.at[idx, "coingecko_id"])
        symbol = str(df.at[idx, "symbol"])

        if top_exchange_cache is not None and coingecko_id in top_exchange_cache:
            df.at[idx, "top_exchange_count"] = top_exchange_cache[coingecko_id]
            continue

        try:
            fetched_value = fetch_top_exchange_count_for_asset(
                coingecko_id=coingecko_id,
                exchange_ids=exchange_ids,
            )
            df.at[idx, "top_exchange_count"] = fetched_value
            fetched_now_count += 1

            if top_exchange_cache is not None:
                top_exchange_cache[coingecko_id] = fetched_value

        except Exception as e:
            df.at[idx, "top_exchange_count"] = pd.NA
            exchange_error_rows.append(
                {
                    "date": df.at[idx, "date"],
                    "coingecko_id": coingecko_id,
                    "symbol": symbol,
                    "error_message": repr(e),
                }
            )

    return df, fetched_now_count


def filter_daily_candidates_by_exchange_count(
    candidate_df: pd.DataFrame,
    min_top_exchange_count: int,
) -> pd.DataFrame:
    df = candidate_df.copy()
    df["top_exchange_count"] = pd.to_numeric(df["top_exchange_count"], errors="coerce")

    exchange_mask = df["top_exchange_count"] >= min_top_exchange_count
    return df.loc[exchange_mask].reset_index(drop=True)


def merge_assets_cumulative(existing_assets: pd.DataFrame, new_asset_core_df: pd.DataFrame) -> pd.DataFrame:
    existing_assets = ensure_asset_columns(existing_assets.copy())
    new_asset_core_df = new_asset_core_df.copy()

    if new_asset_core_df.empty:
        return existing_assets

    new_asset_core_df = new_asset_core_df[
        [
            "coingecko_id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
            "top_exchange_count",
        ]
    ].drop_duplicates(subset=["coingecko_id"], keep="last").reset_index(drop=True)

    if existing_assets.empty:
        new_asset_core_df.insert(0, "asset_id", range(1, len(new_asset_core_df) + 1))
        new_asset_core_df["asset_id"] = new_asset_core_df["asset_id"].astype(int)
        return ensure_asset_columns(new_asset_core_df)

    existing_ids_df = existing_assets[["asset_id", "coingecko_id"]].copy()

    new_with_ids = new_asset_core_df.merge(
        existing_ids_df,
        on="coingecko_id",
        how="left",
    )

    next_asset_id = int(existing_assets["asset_id"].max()) + 1
    missing_mask = new_with_ids["asset_id"].isna()
    missing_count = int(missing_mask.sum())

    if missing_count > 0:
        new_with_ids.loc[missing_mask, "asset_id"] = range(next_asset_id, next_asset_id + missing_count)

    new_with_ids["asset_id"] = new_with_ids["asset_id"].astype(int)

    existing_indexed = existing_assets.set_index("coingecko_id")
    new_indexed = new_with_ids.set_index("coingecko_id")

    cols_to_update = [
        "asset_id",
        "symbol",
        "name",
        "circulating_supply",
        "total_supply",
        "max_supply",
        "top_exchange_count",
    ]
    existing_indexed.update(new_indexed[cols_to_update])

    new_only = new_indexed.loc[~new_indexed.index.isin(existing_indexed.index)].copy()

    combined = pd.concat([existing_indexed, new_only], axis=0).reset_index()
    combined = combined[
        [
            "asset_id",
            "coingecko_id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
            "top_exchange_count",
        ]
    ].copy()

    combined["asset_id"] = combined["asset_id"].astype(int)
    combined = combined.sort_values("asset_id").reset_index(drop=True)

    return ensure_asset_columns(combined)


def attach_asset_ids(
    eligible_daily_df: pd.DataFrame,
    asset_df: pd.DataFrame,
) -> pd.DataFrame:
    df = eligible_daily_df.copy()
    asset_lookup_df = asset_df[["asset_id", "coingecko_id"]].copy()

    df = df.merge(
        asset_lookup_df,
        on="coingecko_id",
        how="left",
    )

    if df["asset_id"].isna().any():
        missing_ids = df.loc[df["asset_id"].isna(), "coingecko_id"].drop_duplicates().tolist()
        raise ValueError(f"Missing asset_id after merge for coingecko_id values: {missing_ids}")

    df["asset_id"] = df["asset_id"].astype(int)
    return df

In [79]:
# daily asset table hlpers
def build_asset_core_df_from_eligible_daily(eligible_daily_df: pd.DataFrame) -> pd.DataFrame:
    df = eligible_daily_df.copy()

    out = df[
        [
            "coingecko_id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
            "top_exchange_count",
        ]
    ].drop_duplicates(subset=["coingecko_id"], keep="last").reset_index(drop=True)

    return out


def build_asset_daily_df_from_eligible_daily(eligible_daily_df: pd.DataFrame) -> pd.DataFrame:
    df = eligible_daily_df.copy()

    df["binance_volume_24h_usd"] = pd.NA

    out = df[
        [
            "date",
            "asset_id",
            "price_usd",
            "market_cap_usd",
            "global_volume_24h_usd",
            "binance_volume_24h_usd",
        ]
    ].copy()

    out = enforce_columns(out, ASSET_DAILY_COLUMNS)
    return out


def append_only_new_asset_daily(existing_df: pd.DataFrame, new_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    existing_df = existing_df.copy()
    new_df = new_df.copy()

    if new_df.empty:
        return existing_df, new_df

    if existing_df.empty:
        combined = new_df.sort_values(["date", "asset_id"]).reset_index(drop=True)
        return combined, new_df.copy()

    existing_keys = set(zip(existing_df["date"].astype(str), existing_df["asset_id"].astype(int)))
    is_new_mask = new_df.apply(lambda r: (str(r["date"]), int(r["asset_id"])) not in existing_keys, axis=1)

    appended_df = new_df.loc[is_new_mask].copy()

    combined = pd.concat([existing_df, appended_df], ignore_index=True)
    combined = combined.sort_values(["date", "asset_id"]).reset_index(drop=True)

    return combined, appended_df


def synchronize_asset_table_to_daily(asset_df: pd.DataFrame, asset_daily_df: pd.DataFrame) -> pd.DataFrame:
    asset_df = ensure_asset_columns(asset_df)
    asset_daily_df = enforce_columns(asset_daily_df, ASSET_DAILY_COLUMNS)

    if asset_daily_df.empty:
        return pd.DataFrame(columns=ASSET_COLUMNS)

    remaining_asset_ids = set(asset_daily_df["asset_id"].astype(int).unique())

    out = (
        asset_df.loc[asset_df["asset_id"].astype(int).isin(remaining_asset_ids)]
        .copy()
        .sort_values("asset_id")
        .reset_index(drop=True)
    )

    return ensure_asset_columns(out)

In [80]:
# processsing helper for one single day
def process_single_run_date(
    run_date: str,
    asset_df: pd.DataFrame,
    combined_asset_daily_df: pd.DataFrame,
    top_exchange_ids: list[str],
    min_volume_24h_usd: float,
    min_top_exchange_count: int,
    top_n: int,
    exchange_error_rows: list[dict],
    top_exchange_cache: dict[str, int | float | pd._libs.missing.NAType] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    daily_top_assets_df = fetch_top_n_universe_for_date(run_date, top_n=top_n)
    daily_universe_df = prepare_daily_universe_df(daily_top_assets_df, date_str=run_date)

    daily_volume_filtered_df = filter_daily_universe_by_volume(
        daily_universe_df=daily_universe_df,
        min_volume_24h_usd=min_volume_24h_usd,
    )

    daily_with_known_exchange_df = attach_known_top_exchange_counts(
        candidate_df=daily_volume_filtered_df,
        asset_df=asset_df,
        top_exchange_cache=top_exchange_cache,
    )

    missing_before_fetch = int(daily_with_known_exchange_df["top_exchange_count"].isna().sum())

    daily_with_exchange_df, fetched_now_count = update_missing_top_exchange_counts_for_candidates(
        candidate_df=daily_with_known_exchange_df,
        exchange_ids=top_exchange_ids,
        exchange_error_rows=exchange_error_rows,
        top_exchange_cache=top_exchange_cache,
    )

    eligible_daily_df = filter_daily_candidates_by_exchange_count(
        candidate_df=daily_with_exchange_df,
        min_top_exchange_count=min_top_exchange_count,
    )

    asset_core_df = build_asset_core_df_from_eligible_daily(eligible_daily_df)

    asset_df = merge_assets_cumulative(
        existing_assets=asset_df,
        new_asset_core_df=asset_core_df,
    )

    if eligible_daily_df.empty:
        day_asset_daily_df = pd.DataFrame(columns=ASSET_DAILY_COLUMNS)
        appended_asset_daily_df = pd.DataFrame(columns=ASSET_DAILY_COLUMNS)
    else:
        eligible_daily_df = attach_asset_ids(
            eligible_daily_df=eligible_daily_df,
            asset_df=asset_df,
        )

        day_asset_daily_df = build_asset_daily_df_from_eligible_daily(eligible_daily_df)

        combined_asset_daily_df, appended_asset_daily_df = append_only_new_asset_daily(
            existing_df=combined_asset_daily_df,
            new_df=day_asset_daily_df,
        )

    asset_df = synchronize_asset_table_to_daily(asset_df, combined_asset_daily_df)

    day_summary = {
        "date": run_date,
        "top_n_rows": len(daily_universe_df),
        "after_volume_filter": len(daily_volume_filtered_df),
        "missing_exchange_counts_before_fetch": missing_before_fetch,
        "exchange_counts_fetched_now": fetched_now_count,
        "after_exchange_filter": len(eligible_daily_df),
        "appended_asset_daily_rows": len(appended_asset_daily_df),
        "total_asset_rows": len(asset_df),
        "total_asset_daily_rows": len(combined_asset_daily_df),
    }

    return asset_df, combined_asset_daily_df, appended_asset_daily_df, day_summary

In [81]:
# Validation helper
def validate_final_tables(asset_df: pd.DataFrame, asset_daily_df: pd.DataFrame) -> None:
    asset_df = ensure_asset_columns(asset_df)
    asset_daily_df = enforce_columns(asset_daily_df, ASSET_DAILY_COLUMNS)

    if asset_df["asset_id"].duplicated().any():
        raise ValueError("Duplicate asset_id found in asset_df.")

    if asset_df["coingecko_id"].duplicated().any():
        raise ValueError("Duplicate coingecko_id found in asset_df.")

    if asset_df["symbol"].isna().any() or (asset_df["symbol"].astype(str).str.strip() == "").any():
        raise ValueError("Empty symbol found in asset_df.")

    if asset_df["name"].isna().any() or (asset_df["name"].astype(str).str.strip() == "").any():
        raise ValueError("Empty name found in asset_df.")

    if asset_daily_df.duplicated(subset=["date", "asset_id"]).any():
        raise ValueError("Duplicate (date, asset_id) found in asset_daily_df.")

    for col in ["price_usd", "market_cap_usd", "global_volume_24h_usd"]:
        if (pd.to_numeric(asset_daily_df[col], errors="coerce").dropna() < 0).any():
            raise ValueError(f"Negative values found in {col}.")

    if not asset_daily_df.empty:
        daily_asset_ids = set(asset_daily_df["asset_id"].astype(int).unique())
        master_asset_ids = set(asset_df["asset_id"].astype(int).unique())

        if not daily_asset_ids.issubset(master_asset_ids):
            raise ValueError("asset_daily_df contains asset_id values not present in asset_df.")

In [82]:
# Build major exchange universe
cmc_top_exchanges_df = fetch_top_exchange_names_from_cmc(TOP_EXCHANGES_N)
coingecko_exchange_map_df = fetch_coingecko_exchange_id_map()

top_exchange_map_df = map_cmc_exchanges_to_coingecko_ids(
    cmc_exchanges_df=cmc_top_exchanges_df,
    coingecko_exchange_map_df=coingecko_exchange_map_df,
)

top_exchange_ids = (
    top_exchange_map_df.loc[
        top_exchange_map_df["coingecko_exchange_id"].notna(),
        "coingecko_exchange_id",
    ]
    .astype(str)
    .tolist()
)

print(f"Top exchange names requested: {TOP_EXCHANGES_N}")
print(f"Mapped CoinGecko exchange ids: {len(top_exchange_ids)}")
display(top_exchange_map_df)

Top exchange names requested: 20
Mapped CoinGecko exchange ids: 19


,exchange_name,name_norm,coingecko_exchange_id,coingecko_exchange_name
0,Binance,binance,binance,Binance
1,Coinbase Exchange,coinbase exchange,gdax,Coinbase Exchange
2,Upbit,upbit,upbit,Upbit
3,OKX,okx,okex,OKX
4,Bybit,bybit,bybit_spot,Bybit
5,Bitget,bitget,bitget,Bitget
6,Gate,gate,gate,Gate
7,KuCoin,kucoin,kucoin,KuCoin
8,MEXC,mexc,mxc,MEXC
9,HTX,htx,huobi,HTX


In [83]:
# Date range
date_list = make_date_list(START_DATE, END_DATE)

print("Date range validated.")
print(f"START_DATE: {START_DATE}")
print(f"END_DATE: {END_DATE}")
print(f"Days to fetch: {len(date_list)}")

Date range validated.
START_DATE: 2026-03-01
END_DATE: 2026-03-31
Days to fetch: 31


In [84]:
# Load existing table
asset_df = load_existing_assets()
combined_asset_daily_df = load_existing_asset_daily()

exchange_count_errors = []
day_summary_rows = []

top_exchange_cache = {}

if not asset_df.empty:
    cached_counts_df = asset_df[["coingecko_id", "top_exchange_count"]].copy()
    cached_counts_df = cached_counts_df.loc[cached_counts_df["top_exchange_count"].notna()].copy()
    top_exchange_cache = dict(
        zip(
            cached_counts_df["coingecko_id"].astype(str),
            pd.to_numeric(cached_counts_df["top_exchange_count"], errors="coerce"),
        )
    )

print(f"Loaded asset rows: {len(asset_df)}")
print(f"Loaded asset_daily rows: {len(combined_asset_daily_df)}")
print(f"Preloaded top_exchange_count cache entries: {len(top_exchange_cache)}")

Loaded asset rows: 45
Loaded asset_daily rows: 2648
Preloaded top_exchange_count cache entries: 45


In [85]:
# Main run loop
for day_idx, run_date in enumerate(date_list, start=1):
    asset_df, combined_asset_daily_df, appended_asset_daily_df, day_summary = process_single_run_date(
        run_date=run_date,
        asset_df=asset_df,
        combined_asset_daily_df=combined_asset_daily_df,
        top_exchange_ids=top_exchange_ids,
        min_volume_24h_usd=MIN_VOLUME_24H_USD,
        min_top_exchange_count=MIN_TOP_EXCHANGE_COUNT,
        top_n=TOP_N,
        exchange_error_rows=exchange_count_errors,
        top_exchange_cache=top_exchange_cache,
    )

    day_summary_rows.append(day_summary)

    ensure_output_dirs()
    asset_df.to_parquet(ASSETS_PATH, index=False)
    combined_asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

    print(
        f"Day {day_idx}/{len(date_list)} | {run_date} | "
        f"top100={day_summary['top_n_rows']} | "
        f"after_volume={day_summary['after_volume_filter']} | "
        f"missing_before_fetch={day_summary['missing_exchange_counts_before_fetch']} | "
        f"fetched_now={day_summary['exchange_counts_fetched_now']} | "
        f"eligible={day_summary['after_exchange_filter']} | "
        f"appended={day_summary['appended_asset_daily_rows']}"
    )

print("Finished processing all dates.")
print(f"Final asset rows so far: {len(asset_df)}")
print(f"Final asset_daily rows so far: {len(combined_asset_daily_df)}")

Day 1/31 | 2026-03-01 | top100=100 | after_volume=50 | missing_before_fetch=5 | fetched_now=5 | eligible=45 | appended=45
Day 2/31 | 2026-03-02 | top100=100 | after_volume=50 | missing_before_fetch=1 | fetched_now=1 | eligible=44 | appended=44
Day 3/31 | 2026-03-03 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 4/31 | 2026-03-04 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 5/31 | 2026-03-05 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 6/31 | 2026-03-06 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 7/31 | 2026-03-07 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 8/31 | 2026-03-08 | top100=100 | after_volume=50 | missing_before_fetch=0 | fetched_now=0 | eligible=44 | appended=44
Day 9/31 | 2026-03-09 | 

In [86]:
# Validation
asset_df = ensure_asset_columns(asset_df)
combined_asset_daily_df = enforce_columns(combined_asset_daily_df, ASSET_DAILY_COLUMNS)

validate_final_tables(asset_df, combined_asset_daily_df)

print("All validation checks passed.")

All validation checks passed.


In [87]:
# Output tables
day_summary_df = pd.DataFrame(day_summary_rows)
exchange_count_error_df = pd.DataFrame(exchange_count_errors)

display(day_summary_df)
display(asset_df)
display(combined_asset_daily_df.sort_values(["date", "asset_id"]).reset_index(drop=True))

if not exchange_count_error_df.empty:
    print("Top-exchange count errors:")
    display(exchange_count_error_df)

,date,top_n_rows,after_volume_filter,missing_exchange_counts_before_fetch,exchange_counts_fetched_now,after_exchange_filter,appended_asset_daily_rows,total_asset_rows,total_asset_daily_rows
0,2026-03-01,100,50,5,5,45,45,45,2693
1,2026-03-02,100,50,1,1,44,44,45,2737
2,2026-03-03,100,50,0,0,44,44,45,2781
3,2026-03-04,100,50,0,0,44,44,45,2825
4,2026-03-05,100,50,0,0,44,44,45,2869
5,2026-03-06,100,50,0,0,44,44,45,2913
6,2026-03-07,100,50,0,0,44,44,45,2957
7,2026-03-08,100,50,0,0,44,44,45,3001
8,2026-03-09,100,50,0,0,44,44,45,3045
9,2026-03-10,100,50,0,0,44,44,45,3089


,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply,top_exchange_count
0,1,bitcoin,BTC,Bitcoin,2.001524e+07,2.001524e+07,2.100000e+07,19.0
1,2,ethereum,ETH,Ethereum,1.206910e+08,1.206910e+08,NaN,19.0
2,3,tether,USDT,Tether,1.844363e+11,1.899033e+11,NaN,19.0
3,4,binancecoin,BNB,BNB,1.363565e+08,1.363565e+08,2.000000e+08,16.0
4,5,ripple,XRP,XRP,6.140553e+10,9.998569e+10,1.000000e+11,19.0
5,6,usd-coin,USDC,USDC,7.861339e+10,7.861782e+10,NaN,18.0
6,7,solana,SOL,Solana,5.745201e+08,6.242546e+08,NaN,19.0
7,8,tron,TRX,TRON,9.477412e+10,9.477425e+10,NaN,16.0
8,9,dogecoin,DOGE,Dogecoin,1.538112e+11,1.538119e+11,NaN,19.0
9,10,hyperliquid,HYPE,Hyperliquid,2.383853e+08,9.622740e+08,1.000000e+09,14.0


,date,asset_id,price_usd,market_cap_usd,global_volume_24h_usd,binance_volume_24h_usd
0,2026-01-01,1,70944.000000,1420084757787,2.863769e+10,None
1,2026-01-01,2,2185.160000,263776706806,1.287374e+10,None
2,2026-01-01,3,1.000000,184451817798,4.660400e+10,None
3,2026-01-01,4,597.880000,81546829453,8.422890e+08,None
4,2026-01-01,5,1.330000,81445184554,1.776721e+09,None
...,...,...,...,...,...,...
4008,2026-03-31,40,0.110815,669416645,6.617386e+07,<NA>
4009,2026-03-31,41,0.859983,663279909,6.828964e+07,<NA>
4010,2026-03-31,42,2.800000,651821982,8.904841e+07,<NA>
4011,2026-03-31,43,41.050000,519225793,1.347318e+08,<NA>


In [88]:
# Save final tables
ensure_output_dirs()

asset_df.to_parquet(ASSETS_PATH, index=False)
combined_asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

print(f"Saved asset table: {ASSETS_PATH}")
print(f"Saved asset_daily table: {ASSET_DAILY_PATH}")
print(f"Total asset rows stored: {len(asset_df)}")
print(f"Total asset_daily rows stored: {len(combined_asset_daily_df)}")

Saved asset table: ../../data/raw/fetched/assets.parquet
Saved asset_daily table: ../../data/raw/fetched/asset_daily.parquet
Total asset rows stored: 45
Total asset_daily rows stored: 4013
